# Q-learning tabular en FrozenLake

Ejemplo de entrenamiento y evaluación separados con la API actual de Gymnasium.

## Dependencias

En un entorno nuevo: `%pip install gymnasium numpy matplotlib`.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=True)
n_states = env.observation_space.n
n_actions = env.action_space.n
q = np.zeros((n_states, n_actions))
rng = np.random.default_rng(2026)

In [ ]:
alpha, gamma = 0.12, 0.99
episodes = 20_000
rewards = np.zeros(episodes)

for episode in range(episodes):
    state, _ = env.reset(seed=episode)
    epsilon = max(0.05, 1.0 - episode / 12_000)
    done = False
    while not done:
        if rng.random() < epsilon:
            action = int(rng.integers(n_actions))
        else:
            action = int(np.argmax(q[state]))
        next_state, reward, terminated, truncated, _ = env.step(action)
        target = reward if terminated else reward + gamma * np.max(q[next_state])
        q[state, action] += alpha * (target - q[state, action])
        state = next_state
        done = terminated or truncated
    rewards[episode] = reward
env.close()

In [ ]:
window = 500
moving_success = np.convolve(rewards, np.ones(window) / window, mode='valid')
plt.plot(np.arange(window - 1, episodes), moving_success)
plt.xlabel('Episodio')
plt.ylabel(f'Tasa de éxito móvil ({window})')
plt.grid(alpha=0.3)
plt.show()

## Evaluación sin exploración

In [ ]:
def evaluar(politica, n=2000, semilla=100_000):
    env_eval = gym.make('FrozenLake-v1', is_slippery=True)
    env_eval.action_space.seed(semilla)
    retornos = []
    for episode in range(n):
        state, _ = env_eval.reset(seed=semilla + episode)
        done, total = False, 0.0
        while not done:
            action = politica(state, env_eval)
            state, reward, terminated, truncated, _ = env_eval.step(action)
            total += reward
            done = terminated or truncated
        retornos.append(total)
    env_eval.close()
    return np.mean(retornos)

greedy = lambda state, _: int(np.argmax(q[state]))
random_policy = lambda state, env: env.action_space.sample()
print('Q-learning:', evaluar(greedy))
print('Aleatoria:', evaluar(random_policy))

## Trabajo propuesto

Compare mapas, descuentos y calendarios de exploración. Reporte media e intervalo de confianza de varias semillas de entrenamiento.